# Augmented SBERT 教學：用「老師模型」幫沒標籤的資料貼標籤

---

## 這個方法想解決什麼問題

做句向量模型（embedding model）時，常遇到：

- **有標籤的資料很少**（人工標註很貴）
- **沒標籤的資料很多**（隨便都找得到）

**Augmented SBERT** 的點子：先用少量有標籤的資料訓練一個「準但慢」的老師，讓老師去幫大量沒標籤的資料貼標籤，最後把兩批資料合在一起，訓練「快」的學生。

```
      10000 筆有標籤資料（gold，金標準）
              │
      ┌───────┴────────┐
      ▼                │
① 訓練 Cross-Encoder   │
   （老師：準但慢）      │
      │                │
      ▼                │
② 老師幫 40000 筆       │
   沒標籤資料貼標籤      │
   （silver，銀標準）    │
      │                │
      └───────┬────────┘
              ▼
   ③ gold + silver 合併 → 約 50000 筆
              │
              ▼
   ④ 訓練 Bi-Encoder（學生：快，能產生句向量）
              │
              ▼
   ⑤ 用 STS-B 評估
```

## 兩種模型的差別

| | Cross-Encoder（老師） | Bi-Encoder（學生） |
|---|---|---|
| 輸入方式 | 兩個句子**一起**丟進 BERT | 兩個句子**各自**丟進 BERT |
| 輸出 | 一個判斷（例如 0 或 1） | 每個句子一個向量 |
| 準確度 | 較高（兩句能互相參照） | 較低 |
| 速度 | 慢（每一對都要重算） | 快（向量可預先存好） |
| 能不能產生句向量 | **不能** | **能**（這才是我們要的） |

所以老師只負責「幫忙貼標籤」，最後真正要用的是學生。

## 步驟 0：載入套件

幾個重點套件：

| 套件 / 類別 | 用途 |
|---|---|
| `load_dataset`、`Dataset` | 載入並建立 Hugging Face 資料集 |
| `SentenceTransformer` | Bi-Encoder（學生模型） |
| `losses` | 各種損失函數，這裡會用 `CosineSimilarityLoss` |
| `EmbeddingSimilarityEvaluator` | 用句子相似度資料集評估模型 |
| `InputExample`、`NoDuplicatesDataLoader` | 舊版訓練 Cross-Encoder 所需的資料容器（下面會解釋） |
| `pandas` | 用來合併 gold 和 silver 兩批資料 |

In [1]:
from datasets import load_dataset, Dataset
from sentence_transformers.sentence_transformer.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.sentence_transformer import losses
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.trainer import SentenceTransformerTrainer
from tqdm import tqdm
import pandas as pd
from sentence_transformers import InputExample
from sentence_transformers.sentence_transformer.datasets import NoDuplicatesDataLoader

## 步驟 1：載入 gold 資料並定義標籤對照

```python
.select(range(10000))
```

只取 MNLI 訓練集的**前 10000 筆**當作 gold（有人工標籤的資料）。後面的 10000~50000 筆會拿來當 silver。**兩批資料互不重疊**，這點很重要。

### MNLI 的資料長什麼樣

每筆資料有 `premise`（前提）、`hypothesis`（假設）、`label`：

| label | 意思 | 例子 |
|---|---|---|
| 0 | 蘊含（entailment） | 前提：一個男人在騎馬 → 假設：有人在騎動物 |
| 1 | 中立（neutral） | 前提：一個男人在騎馬 → 假設：男人在比賽 |
| 2 | 矛盾（contradiction） | 前提：一個男人在騎馬 → 假設：男人在睡覺 |

### `mapping = {0:1, 1:0, 2:0}` 是什麼

這是一張**對照表**（dict），把三類標籤簡化成兩類：

```
0（蘊含）→ 1（相似）
1（中立）→ 0（不相似）
2（矛盾）→ 0（不相似）
```

為什麼要轉？因為後面訓練 Bi-Encoder 用的是 `CosineSimilarityLoss`，它要的是「相似度」，不是「類別代號」。如果直接餵 0/1/2，數字大小會被誤解成相似程度，語意會亂掉。

> 注意：把「中立」也當成「不相似」是一個**設計選擇**，不是唯一答案。

In [2]:
dataset = load_dataset(
    "nyu-mll/glue",
    "mnli",
    split ="train",
).select(range(10000))
print(dataset[0])

mapping = {0:1, 1:0, 2:0}

{'premise': 'Conceptually cream skimming has two basic dimensions - product and geography.', 'hypothesis': 'Product and geography are what make cream skimming work. ', 'label': 1, 'idx': 0}


## 步驟 2：把 gold 準備成兩種容器

這一段會讓初學者最困惑：**同一份 gold 資料，為什麼準備兩次？**

答案：後面有兩個「使用者」，各自只吃一種格式。

| 容器 | 給誰用 | 原因 |
|---|---|---|
| `gold_examples` → `gold_dataloader` | 訓練 Cross-Encoder（老師） | 舊版 `cross_encoder.fit()` 要吃 `DataLoader`，裡面要裝 `InputExample` |
| `gold`（pandas DataFrame） | 之後跟 silver 合併，訓練 Bi-Encoder（學生） | 合併、去重複用 pandas 最方便 |

### InputExample

舊版 sentence-transformers 規定的「一筆資料」格式：

```python
InputExample(texts=["句子A", "句子B"], label=1)
```

### DataLoader 與 NoDuplicatesDataLoader

DataLoader 像「發牌員」：把全部資料交給它，它每次拿出一批（這裡是 32 筆）給模型。

`NoDuplicatesDataLoader` 是特別版發牌員，**保證同一批裡沒有重複的句子**。

這個設計主要是為了「對比學習」：同一批的其他句子會被當成負例，如果同一句話出現兩次，就會產生**假負例**（把相同的句子要求推開），傷害訓練。

> 老實說：這裡訓練的是 Cross-Encoder 做**分類**，每一對句子獨立判斷，不會拿同批句子當負例，所以其實不一定需要它。教學範例沿用這個慣例，無害，只是不是必需品。

### 這段還建立了 `gold` DataFrame

欄位固定為 `sentence1`、`sentence2`、`label`，標籤已經用 `mapping` 轉成 0/1。等一下 silver 也會用**同樣的欄位格式**，兩個表才能直接疊在一起。

In [3]:
gold_examples = [
    InputExample(
        texts = [row["premise"], row["hypothesis"]],
        label = mapping[row["label"]],
    ) for row in tqdm(dataset)
]
print(gold_examples[0])

gold_dataloader = NoDuplicatesDataLoader(gold_examples, batch_size = 32)

gold = pd.DataFrame(
    {
        "sentence1" : dataset["premise"],
        "sentence2" : dataset["hypothesis"],
        "label" : [mapping[label] for label in dataset["label"]],
    }
)
print(gold.head())

100%|██████████| 10000/10000 [00:00<00:00, 100883.30it/s]

<InputExample> label: 0, texts: Conceptually cream skimming has two basic dimensions - product and geography.; Product and geography are what make cream skimming work. 


                                           sentence1  \
0  Conceptually cream skimming has two basic dime...   
1  you know during the season and i guess at at y...   
2  One of our number will carry out your instruct...   
3  How do you know? All this is their information...   
4  yeah i tell you what though if you go price so...   

                                           sentence2  label  
0  Product and geography are what make cream skim...      0  
1  You lose the things to the following level if ...      1  
2  A member of my team will execute your orders w...      1  
3                  This information belongs to them.      1  
4           The tennis shoes have a range of prices.      0  


## 步驟 3：訓練 Cross-Encoder（老師）

```python
CrossEncoder("bert-base-uncased", num_labels=2)
```

- 以 `bert-base-uncased` 為底座，在上面加一個分類頭
- `num_labels=2`：二元分類（0 = 不相似，1 = 相似）
- 輸入是**一對句子一起**，輸出是兩個類別的判斷

### `fit` 的參數

| 參數 | 意思 |
|---|---|
| `train_dataloader` | 剛剛準備的 `gold_dataloader` |
| `epochs=1` | 整份資料看一遍 |
| `show_progress_bar=True` | 顯示進度條 |
| `warmup_steps=100` | 前 100 步學習率由小慢慢加大，避免一開始更新太劇烈 |
| `use_amp=False` | 不使用混合精度（fp16 / bf16）。要加速可以改 `True`，需 GPU 支援 |

大約的步數：10000 筆 / 32 ≈ 313 步（`NoDuplicatesDataLoader` 可能略少）。

> **版本提醒**：`cross_encoder.fit(train_dataloader=...)` 搭配 `InputExample` 是**舊版 API**。新版的 Cross-Encoder 改為 Trainer 風格、直接吃 `Dataset`。我沒有針對你安裝的版本驗證，若這一格報錯，優先檢查是不是新舊 API 的差異。

In [4]:
from sentence_transformers.cross_encoder import CrossEncoder

cross_encoder = CrossEncoder(
    "bert-base-uncased",
    num_labels = 2,
)
cross_encoder.fit(
    train_dataloader = gold_dataloader,
    epochs = 1,
    show_progress_bar = True,
    warmup_steps = 100,
    use_amp = False, 
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


[W1003 15:55:00.461253870 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 98566144 bytes (free: 35258368, total: 8082096128).


## 步驟 4：載入 silver 資料（沒標籤的資料）

```python
.select(range(10000, 50000))
```

取第 10000 到 50000 筆，共 **40000 筆**。雖然 MNLI 本來有標籤，但這裡**假裝沒有**，不會使用它們，改由老師來預測。

這就是 Augmented SBERT 的精神：實務上，silver 通常是真正沒標籤的資料。

```python
pairs = list(zip(silver["premise"], silver["hypothesis"]))
```

`zip` 把兩個欄位並排，轉成 `[(前提1, 假設1), (前提2, 假設2), ...]` 的形式，這是 Cross-Encoder `predict` 要的輸入格式。

In [5]:
silver = load_dataset(
    "nyu-mll/glue",
    "mnli",
    split ="train",
).select(range(10000, 50000))
pairs = list(zip(silver["premise"], silver["hypothesis"]))
print(pairs[0])

('Hindus and Buddhists still bathe where he bathed.', 'Hindus and Buddhists bathe in the same location.')


## 步驟 5：老師幫 silver 貼標籤

```python
output = cross_encoder.predict(pairs, apply_softmax=True)
```

- 老師對每一對句子輸出**兩個機率**，`apply_softmax=True` 讓它們加起來等於 1
- `output` 的形狀是 `(40000, 2)`

例如某一筆：

```
output[i] = [0.13, 0.87]
              ↑     ↑
        不相似的機率  相似的機率
```

```python
np.argmax(output, axis=1)
```

`argmax` 取機率較大的那一欄的**索引**：`[0.13, 0.87]` → `1`。也就是把機率轉成**硬標籤**（只剩 0 或 1）。

最後把結果組成跟 gold 一樣欄位的 DataFrame（`sentence1`、`sentence2`、`label`）。

> 這裡有個值得思考的點：`argmax` 會把「0.51 vs 0.49」和「0.99 vs 0.01」都變成同樣的標籤，**丟掉了老師有多確定的資訊**。這是目前這份程式的設計，不是錯誤；若想保留這個資訊，可以改用機率當標籤（soft label），見最後的「可以自己做的實驗」。

In [6]:
import numpy as np

output = cross_encoder.predict(
    pairs,
    apply_softmax = True,
)
print(output)
silver = pd.DataFrame(
    {
        "sentence1" : silver["premise"],
        "sentence2" : silver["hypothesis"],
        "label" : np.argmax(output, axis = 1)
    }
)

[[0.4006224  0.5993776 ]
 [0.95472074 0.04527924]
 [0.9587703  0.04122974]
 ...
 [0.92860633 0.07139364]
 [0.19262037 0.8073796 ]
 [0.18514045 0.8148596 ]]


## 步驟 6：合併 gold + silver，建立訓練資料集

```python
pd.concat([gold, silver], axis=0, ignore_index=True)
```

- `axis=0`：上下疊起來（把 silver 接在 gold 下面）
- `ignore_index=True`：重新編號，不沿用兩個表原本的 index

```python
data.drop_duplicates(subset=["sentence1", "sentence2"], keep="first")
```

- 如果同一組 `(sentence1, sentence2)` 出現多次，只留第一筆
- 因為 gold 排在前面，**真的重複時會優先保留 gold（人工標籤）**，這是 `keep="first"` 的好處

```python
Dataset.from_pandas(data, preserve_index=False)
```

把 pandas DataFrame 轉成 Hugging Face 的 `Dataset`（底層是 Arrow 格式，讀取快、省記憶體）。`preserve_index=False` 不把 index 當成額外欄位帶進去。

合併後約 50000 筆，其中：

| 來源 | 筆數 | 標籤來源 | 可信度 |
|---|---|---|---|
| gold | 10000 | 人工標註 | 高 |
| silver | 40000 | 老師預測 | 取決於老師有多準 |

In [7]:
data = pd.concat(
    [gold, silver],
    axis = 0,
    ignore_index = True,
)
data = data.drop_duplicates(
    subset=["sentence1", "sentence2"],
    keep = "first",
)
train_dataset = Dataset.from_pandas(
    data,
    preserve_index = False
)
print(train_dataset[0])

{'sentence1': 'Conceptually cream skimming has two basic dimensions - product and geography.', 'sentence2': 'Product and geography are what make cream skimming work. ', 'label': 0}


## 步驟 7：建立評估器（STS-B）

STS-B 是另一個資料集，每對句子有人工標註的相似度分數（0~5）。

```python
scores = [score / 5 for score in val_sts["label"]]
```

除以 5，把分數正規化到 0~1。

`EmbeddingSimilarityEvaluator` 的流程：

1. 用模型把每對句子轉成向量
2. 計算兩個向量的餘弦相似度（`main_similarity="cosine"`）
3. 和人工分數計算相關係數，**越高越好**

**為什麼用 STS-B 評估？** 訓練用的是 MNLI（蘊含分類），評估用的是 STS-B（相似度），是**沒訓練過的任務**，可以檢驗模型是否真的學到通用的語意向量，而不是只會背訓練資料。

In [8]:
val_sts = load_dataset(
    "nyu-mll/glue",
    "stsb",
    split = "validation",
)
evaluator = EmbeddingSimilarityEvaluator(
sentences1=val_sts["sentence1"],
sentences2=val_sts["sentence2"],
scores = [score / 5 for score in val_sts["label"]],
main_similarity="cosine",
)

## 步驟 8：訓練 Bi-Encoder（學生）並儲存

### 模型與 loss

```python
embedding_model = SentenceTransformer("bert-base-uncased")
train_loss = losses.CosineSimilarityLoss(model=embedding_model)
```

`CosineSimilarityLoss` 的邏輯：

1. 兩個句子各自編碼成向量 u、v
2. 算餘弦相似度 cos(u, v)
3. 和標籤（0 或 1）計算 MSE（均方誤差）

```
標籤 = 1 → 要求 cos(u, v) 接近 1（拉近）
標籤 = 0 → 要求 cos(u, v) 接近 0（不相關）
```

這是**回歸**，所以標籤必須是「有大小意義的數值」，這就是步驟 1 為什麼要 `mapping`。

### 訓練參數

| 參數 | 值 | 說明 |
|---|---|---|
| `output_dir` | `Augmented_SBERT_model` | 輸出資料夾（checkpoint 等） |
| `num_train_epochs` | 1 | 資料看一遍 |
| `per_device_train_batch_size` | 32 | 每步 32 筆；約 50000 / 32 ≈ 1563 步 |
| `warmup_steps` | 100 | 學習率前 100 步從 0 升到預設的 `5e-5` |
| `fp16` | True | 混合精度，省顯存、加速（需要支援的 GPU） |
| `eval_steps` / `logging_steps` | 100 | 每 100 步評估 / 記錄 |

### 訓練前後對比

```python
print("before:", evaluator(embedding_model))
trainer.train()
print("after:", evaluator(embedding_model))
```

`trainer.train()` 會**原地修改** `embedding_model`，所以兩次分數的差，就是訓練帶來的進步。未訓練的 BERT 直接當句向量，分數通常偏低。

### 儲存

```python
embedding_model.save("Augmented_SBERT_model")
```

存的是最終的 BERT + pooling（不含 loss 附帶的東西）。之後可以用 `SentenceTransformer("Augmented_SBERT_model")` 直接載入，不必重新訓練。

> 這個路徑和 `output_dir` 是同一個資料夾，所以最終模型檔會和 `checkpoint-xxx` 子資料夾放在一起。這樣能用，只是檔案會比較混雜。

In [9]:
embedding_model = SentenceTransformer("bert-base-uncased")

train_loss = losses.CosineSimilarityLoss(
    model = embedding_model,
)

args = SentenceTransformerTrainingArguments(
    output_dir = "Augmented_SBERT_model",
    num_train_epochs = 1,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 32,
    warmup_steps = 100,
    fp16 = True,
    eval_steps = 100,
    logging_steps = 100,
)

trainer = SentenceTransformerTrainer(
    model = embedding_model,
    args = args,
    train_dataset = train_dataset,
    loss = train_loss,
    evaluator = evaluator,
)
print("before:", evaluator(embedding_model))
trainer.train()
print("after:", evaluator(embedding_model))

embedding_model.save("Augmented_SBERT_model")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

before: {'pearson_cosine': 0.5917194765781003, 'spearman_cosine': 0.5931748194986226}


Step,Training Loss
100,0.217193
200,0.158495
300,0.145769
400,0.145715
500,0.143959
600,0.138037
700,0.135192
800,0.133862
900,0.138668
1000,0.132112


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

after: {'pearson_cosine': 0.7260400719370693, 'spearman_cosine': 0.7346555176697703}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

---

# 初學者重點整理

## 一、整個方法一句話

> 用少量人工標籤訓練「準但慢」的 Cross-Encoder，讓它幫大量沒標籤的資料貼標籤，再把兩批資料合併，訓練「快」的 Bi-Encoder 當最終的 embedding model。

## 二、必須搞懂的 6 個概念

| # | 概念 | 一句話解釋 |
|---|---|---|
| 1 | gold / silver | gold = 人工標籤（可信），silver = 模型預測標籤（可能有錯） |
| 2 | Cross-Encoder | 兩句一起看，準但慢，不產生句向量，只當老師 |
| 3 | Bi-Encoder | 兩句各自變向量，快，這才是最終要的模型 |
| 4 | `mapping` | 把 MNLI 三類標籤轉成「相似 = 1、不相似 = 0」，因為 `CosineSimilarityLoss` 要的是相似度數值 |
| 5 | `CosineSimilarityLoss` | 讓向量的餘弦相似度逼近標籤，是回歸任務 |
| 6 | STS-B 評估 | 用沒訓練過的相似度資料集，檢驗向量品質 |

## 三、程式中最容易搞混的地方

1. **gold 為什麼準備兩份？**
   `gold_dataloader` 給舊版 `fit()` 訓練老師；`gold` DataFrame 給之後合併。同一份資料，兩種格式。

2. **`NoDuplicatesDataLoader` 是必要的嗎？**
   對這裡的 Cross-Encoder 分類任務**不是必要**。它主要是為了防止對比學習（如 MNR Loss）出現「假負例」。

3. **老師預測的是什麼？**
   `output` 是機率（形狀 `(40000, 2)`），`argmax` 才把它轉成 0/1。

4. **silver 和 gold 為什麼可以直接 `concat`？**
   因為兩邊欄位名稱一樣：`sentence1`、`sentence2`、`label`。

5. **`train_loss` 的 `model` 和 Trainer 的 `model` 是同一個物件**
   所以訓練會原地更新 `embedding_model`。

## 四、資料量與步數速算

| 階段 | 資料量 | 約略步數 |
|---|---|---|
| 訓練 Cross-Encoder | 10000 | 約 313 |
| 老師預測 silver | 40000 | （只推論，不訓練） |
| 訓練 Bi-Encoder | 約 50000 | 約 1563 |

## 五、觀察到的幾個潛在問題（程式碼未修改，僅供參考）

這些不一定是錯誤，是進階時可以思考或實驗的地方：

1. **`eval_steps=100` 不會生效**
   只設 `eval_steps` 沒有設 `eval_strategy="steps"`，訓練過程中預設不會評估。目前只有訓練前後各評估一次。

2. **silver 用硬標籤（argmax）**
   丟掉了老師的確定程度，且老師只用 10000 筆、1 epoch 訓練，不一定很準。

3. **標籤型別是整數**
   `CosineSimilarityLoss` 做回歸，通常用 float 較保險。多數情況整數也能跑。

4. **「中立 = 不相似」是設計選擇**
   中立其實介於相似與不相似之間，可以嘗試給 0.5。

5. **沒有驗證 silver 的品質**
   不知道老師準不準，silver 裡的錯誤標籤會直接進入學生的訓練資料。

6. **版本相容性**
   `cross_encoder.fit` + `InputExample` + `DataLoader` 是舊版寫法，不同版本的 sentence-transformers 可能行為不同。

## 六、可以自己做的實驗（需要時再改，不影響原程式）

```python
# 1. 老師準不準？在 MNLI 驗證集上量一下
val = load_dataset("nyu-mll/glue", "mnli", split="validation_matched").select(range(2000))
val_pairs = list(zip(val["premise"], val["hypothesis"]))
pred = np.argmax(cross_encoder.predict(val_pairs, apply_softmax=True), axis=1)
true = [mapping[l] for l in val["label"]]
print("老師準確率:", np.mean(pred == np.array(true)))

# 2. 用機率當 soft label（保留老師的確定程度）
# "label": output[:, 1]

# 3. 比較「只用 gold」和「gold + silver」兩個版本，看 silver 到底有沒有幫助
```

## 七、一頁速記

```
1. 載入 gold（10000 筆）+ mapping（三類 → 兩類）
2. gold 準備兩份：DataLoader（給老師）、DataFrame（給合併）
3. 訓練 Cross-Encoder 老師
4. 載入 silver（40000 筆，假裝沒標籤）
5. 老師預測 → argmax → silver 標籤
6. gold + silver 合併、去重複 → Dataset
7. 建立 STS-B 評估器
8. 訓練 Bi-Encoder（CosineSimilarityLoss）→ 前後評估 → 儲存
```